# Galaxy morphology classification with AstroLens GCNN

A tutorial-scale training example for `astrolens.models.gcnn.GCNN`
(Pandya et al., 2023, https://arxiv.org/abs/2311.01500), a group-equivariant
CNN, using [`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10) —
a MultimodalUniverse-formatted copy of **Galaxy10 DECals** (17,736 galaxies,
10 discrete morphology classes), the same dataset the paper's own
[reference implementation](https://github.com/snehjp2/GCNNMorphology) trains on,
split 70/10/20 train/val/test.

Optimizer, schedule, and augmentation follow the reference implementation's
`D8.yaml` / `train.py`: `AdamW` (`lr=1e-2`, `weight_decay=1e-4`), `MultiStepLR`
decaying by `0.1`, and the same rotation/affine/flip augmentation. Model,
batch size, and epoch count are set in the cells below. To reproduce the
paper's reported results directly, follow the reference implementation and
its `src/config/*.yaml` files.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision scikit-learn

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import astrolens
from utils import gz10

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and split 70/10/20

Galaxy10 DECals' standard 10 classes (astroNN convention): disturbed, merging,
round smooth, in-between round smooth, cigar-shaped smooth, barred spiral,
unbarred tight spiral, unbarred loose spiral, edge-on without bulge, edge-on
with bulge.

In [ ]:
IMG_SIZE = 255  # GCNN.img_size: sizes the MaskModule's inscribed-circle mask
BATCH_SIZE = 32  # reference uses 128; doesn't fit at 255x255 with group-equivariant conv here
CLASS_NAMES = gz10.CLASS_NAMES
NUM_CLASSES = gz10.NUM_CLASSES
NUM_WORKERS = 4

data, labels, train_idx, val_idx, test_idx = gz10.load_split_702010()

# reference implementation's augmentation and normalization
# (src/scripts/train.py): heavy rotation/flip augmentation exercises the
# model's rotation/reflection equivariance during training.
train_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.RandomRotation(180),
        transforms.Resize(IMG_SIZE),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(IMG_SIZE),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)

train_dataset = gz10.GZ10Dataset(data, train_idx, train_transform)
val_dataset = gz10.GZ10Dataset(data, val_idx, eval_transform)
test_dataset = gz10.GZ10Dataset(data, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

len(train_dataset), len(val_dataset), len(test_dataset)

## Create the model

`gcnn_d4`: dihedral group `D4` (4 rotations x reflection, group order 8).
The reference implementation's own config is `D8` (order 16, `D8.yaml`);
`D4` is used here to keep a full 100-epoch run practical on this GPU
(D8 runs ~2.7x slower per step at this resolution).

In [4]:
model = astrolens.create_model(
    "gcnn_d4",
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,
).to(device)

sum(p.numel() for p in model.parameters())

4598030

## Train

`AdamW`, `MultiStepLR` decaying by `0.1`, and class-weighted
cross-entropy (inverse-frequency weights from the train split, following the
same approach as the Linformer example) to counter GZ10's class imbalance —
the reference implementation itself uses unweighted cross-entropy.
Milestones are rescaled from the reference's `[25, 50, 75]` at 100 epochs to
the same 25/50/75% points of this notebook's shorter run.

In [ ]:
MAX_EPOCHS = 10
LR = 1e-2
WEIGHT_DECAY = 1e-4
MILESTONES = [round(MAX_EPOCHS * f) for f in (0.25, 0.5, 0.75)]  # reference: [25, 50, 75] at 100 epochs
GAMMA = 0.1

# inverse-frequency class weights from the train split, following the same
# approach as the Linformer example, to counter GZ10's class imbalance
criterion = nn.CrossEntropyLoss(weight=gz10.class_weights(labels, train_idx).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = gz10.run_classification_epoch(
        model, train_loader, criterion, device, train=True, optimizer=optimizer
    )
    val_loss, val_acc = gz10.run_classification_epoch(model, val_loader, criterion, device, train=False)
    scheduler.step()
    print(
        f"epoch {epoch}/{MAX_EPOCHS} "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f} "
        f"lr={scheduler.get_last_lr()[0]:.2e}"
    )

## Evaluate on the held-out test split

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_acc=0.537 test_f1_macro=0.509

                         precision    recall  f1-score   support

              disturbed       0.26      0.39      0.32       216
                merging       0.66      0.38      0.48       371
           round_smooth       0.73      0.80      0.76       529
in_between_round_smooth       0.58      0.66      0.62       405
    cigar_shaped_smooth       0.19      0.64      0.29        67
          barred_spiral       0.39      0.33      0.35       409
  unbarred_tight_spiral       0.42      0.48      0.45       366
  unbarred_loose_spiral       0.49      0.26      0.34       525
       edge_on_no_bulge       0.70      0.79      0.74       285
     edge_on_with_bulge       0.74      0.74      0.74       375

               accuracy                           0.54      3548
              macro avg       0.52      0.55      0.51      3548
           weighted avg       0.56      0.54      0.53      3548



## Save the trained weights

Saved for reuse by `examples/gz10_gcnn_analysis.ipynb` (one-pixel attack and latent-space analysis).

In [7]:
CHECKPOINT_PATH = "gcnn_d4.pt"
torch.save(model.state_dict(), CHECKPOINT_PATH)
CHECKPOINT_PATH

'gcnn_d4.pt'